#### Librerias

In [175]:
import pandas as pd
import os
import re
import unicodedata

#### Carga de datos

In [176]:
def cargar_y_concatenar(ruta_carpeta, tipo):
    dfs = []

    for archivo in os.listdir(ruta_carpeta):
        if archivo.endswith(".csv"):

            # Match robusto por tipo + año
            match = re.search(rf"{tipo}_(20\d{{2}})", archivo, re.IGNORECASE)

            if match:
                ruta = os.path.join(ruta_carpeta, archivo)
                año = int(match.group(1))

                try:
                    df = pd.read_csv(
                        ruta,
                        sep=";",
                        encoding="cp1252",
                        encoding_errors="replace",
                        engine="python",
                        on_bad_lines="skip"
                    )

                    df["año"] = año
                    dfs.append(df)

                except Exception as e:
                    print(f"[ERROR] {archivo}: {e}")

    if not dfs:
        raise ValueError(f"No se encontraron archivos para: {tipo}")

    return pd.concat(dfs, ignore_index=True)

In [177]:
ruta = "../../data"

df_viajeros = cargar_y_concatenar(ruta, "viajeros")
df_pernoctaciones = cargar_y_concatenar(ruta, "Pernoctaciones")

#### Revisión de datos

In [178]:
df_viajeros.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6900 entries, 0 to 6899
Data columns (total 5 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   Comunidades y Ciudades Autónomas  6900 non-null   object
 1   País de residencia                6900 non-null   object
 2   Meses                             6900 non-null   object
 3   Total                             6900 non-null   object
 4   año                               6900 non-null   int64 
dtypes: int64(1), object(4)
memory usage: 269.7+ KB


In [179]:
# Convertir nombres de columnas a minúsculas y eliminar espacios
df_viajeros.columns = df_viajeros.columns.str.strip().str.lower()
df_pernoctaciones.columns = df_pernoctaciones.columns.str.strip().str.lower()

In [180]:
df_pernoctaciones.columns

Index(['comunidades y ciudades autónomas', 'país de residencia', 'meses',
       'total', 'año'],
      dtype='object')

In [181]:
# Renombrar columnas para mayor claridad del dato que contienen
df_viajeros.rename(columns={
    "meses": "mes",
    "comunidades y ciudades autónomas": "ccaa",
    "país de residencia": "pais_origen",
    "total":"qty_viajeros"
}, inplace=True)

df_pernoctaciones.rename(columns={
    "meses": "mes",
    "comunidades y ciudades autónomas": "ccaa",
    "país de residencia": "pais_origen",
    "total":"qty_viajeros"
}, inplace=True)


In [182]:
df_viajeros['pais_origen'].value_counts()

pais_origen
Españoles           300
Alemania            300
Austria             300
Bélgica             300
Dinamarca           300
Finlandia           300
Francia             300
Grecia              300
Irlanda             300
Italia              300
Luxemburgo          300
Países Bajos        300
Polonia             300
Portugal            300
República Checa     300
Suecia              300
Noruega             300
Reino Unido         300
Suiza               300
Japón               300
República China     300
Estados Unidos      300
Países africanos    300
Name: count, dtype: int64

#### Transformación

In [183]:
# Convertir columnas de total a numéricas
if df_viajeros["qty_viajeros"].dtype == "object":
    df_viajeros["qty_viajeros"] = pd.to_numeric(
        df_viajeros["qty_viajeros"]
            .str.replace(r"\.", "", regex=True)
            .str.replace(",", ".", regex=False),
        errors="coerce"
    )

if df_pernoctaciones["qty_viajeros"].dtype == "object":
    df_pernoctaciones["qty_viajeros"] = pd.to_numeric(
        df_pernoctaciones["qty_viajeros"]
            .str.replace(r"\.", "", regex=True)
            .str.replace(",", ".", regex=False),
        errors="coerce"
    )

In [184]:
def normalizar(texto):
    if pd.isna(texto):
        return texto
    texto = texto.strip().lower()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("utf-8")
    return texto

# Lista de agregados (en formato normalizado)
agregados = {
    "total",
    "No residentes en España",
    "unión europea 27 (sin españa)",
    "Resto de la UE",
    "resto de america",
    "extranjero",
    "ue27",
    "ue28"
}

def eliminar_agregados(df, columna):
    mask = df[columna].apply(normalizar).isin(agregados)
    return df[~mask]

# Aplicar a ambos dataframes
df_viajeros = eliminar_agregados(df_viajeros, "pais_origen")
df_pernoctaciones = eliminar_agregados(df_pernoctaciones, "pais_origen")

df_pernoctaciones

,ccaa,pais_origen,mes,qty_viajeros,año
0,01 Andalucía,Españoles,Enero,242571.0,2015
1,01 Andalucía,Españoles,Febrero,249795.0,2015
2,01 Andalucía,Españoles,Marzo,311375.0,2015
3,01 Andalucía,Españoles,Abril,273214.0,2015
4,01 Andalucía,Españoles,Mayo,335422.0,2015
...,...,...,...,...,...
6583,"13 Madrid, Comunidad de",Resto del mundo,Agosto,17082.0,2019
6584,"13 Madrid, Comunidad de",Resto del mundo,Septiembre,23970.0,2019
6585,"13 Madrid, Comunidad de",Resto del mundo,Octubre,21953.0,2019
6586,"13 Madrid, Comunidad de",Resto del mundo,Noviembre,20969.0,2019


In [185]:
df_pernoctaciones['pais_origen'].value_counts()

pais_origen
Españoles           300
Alemania            300
Austria             300
Bélgica             300
Dinamarca           300
Finlandia           300
Francia             300
Grecia              300
Irlanda             300
Italia              300
Luxemburgo          300
Países Bajos        300
Polonia             300
Portugal            300
Republica Checa     300
Suecia              300
Noruega             300
Reino Unido         300
Suiza               300
Estados Unidos      300
Países Africanos    300
Resto del mundo     240
Extranjeros          48
Name: count, dtype: int64